**Group information**

| Family name | First name | Email address |
| ----------- | ---------- | ------------- |
|             |            |               |
|             |            |               |
|             |            |               |

<div style="display:flex">
    <img src="https://bse.eu/sites/default/files/bse_logo_small.png" alt="Logo 1">
</div>

# Step 0: set up your virtual environment

If usefull, follow the [DIUx-xView/xView2_baseline Github repository environment-setup](https://github.com/DIUx-xView/xView2_baseline#environment-setup) to create a virtual environment (not needed if you run it on Google Colab) and install the packages below:

In [ ]:
#Installation of the required packages
!pip install numpy matplotlib tqdm scipy Pillow scikit-image opencv-python
!pip install imgaug IPython geopandas keras imantics simplification scikit-learn
!pip install chainer cup
# Uninstall current pandas completely
!pip uninstall -y pandas

# Install specific pandas version with force-reinstall
!pip install --force-reinstall --no-deps pandas==2.2.3

## Imports

In [ ]:
import re
import os, sys
from imutils import paths
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
import numpy as np
import imageio

# Step 1: Download the data

While downloading the data below from tier1.zip, read carefully the documentation of the [xView2: Assess Building Damage](https://xview2.org/dataset) and [Gupta et al](https://arxiv.org/pdf/1911.09296.pdf).

The xBD dataset follows this structure:

```  
xBD
 ├── tier1
 │      ├── images
 │      │      └── <image_id>.png
 │      │      └── ...
 │      ├── labels
 │      │      └── <image_id>.json
 │      │      └── ...
 ├── hold
 │      ├── images
 │      │      └── <image_id>.png
 │      │      └── ...
 │      ├── labels
 │      │      └── <image_id>.json
 │      │      └── ...
 ├── test
 │      ├── images
 │      │      └── <image_id>.png
 │      │      └── ...
 │      ├── labels
 │      │      └── <image_id>.json
 │      │      └── ...
 └```
```
to avoid additional heavy downloading, we currently focus only on `/tier1` (it takes a bit of time to run given the size of the dataset).

In [ ]:
# download the tier1 part
!wget -O tier1.zip https://www.dropbox.com/scl/fi/pkohmx034xpmcix3w7gf4/tier1.zip?rlkey=ckleil15nslwuc3irvyiqtpfr&st=rbbfdlvb&dl=0
!mkdir xBD
!unzip tier1.zip -d xBD/tier1/
!rm tier1.zip

Download the `utils` function:

In [ ]:
# download the utils functions
!wget -O utils.zip https://www.dropbox.com/scl/fi/ofwidtxrasnduljk9i7z0/utils.zip?rlkey=v9y0b0fmftw5y8z3kz2yvbbu2&st=krwaqpze&dl=0
!mkdir utils
!unzip utils.zip -d utils/
!rm utils.zip


## Data preparation

1. Run `utils/split_into_disasters.py` on folder `/tier1` to split files under a single directory (with images/ and labels/ into directory of disasters/images|labels) like this:
```  
xBD
 ├── disaster_name_1
 │      ├── images
 │      │      └── <image_id>.png
 │      │      └── ...
 │      ├── labels
 │      │      └── <image_id>.json
 │      │      └── ...
 ├── disaster_name_2
 │      ├── images
 │      │      └── <image_id>.png
 │      │      └── ...
 │      ├── labels
 │      │      └── <image_id>.json
 │      │      └── ...
 └── disaster_name_n
```  
Check the parameter required by calling:

`!python /content/utils/utils/split_into_disasters.py --help`

In [ ]:
### CODE HERE -

We can now delete the `tier1` to save space:

In [ ]:
!rm -r /content/xBD/tier1

2. Run the `utils/generate_annotation_masks.py` to generate a mask file for the chipped images given the json file in the /labels folder.

Check the parameter required by calling:

`$ python generate_annotation_masks.py --help` (it is a slow process).

In [ ]:
### CODE HERE

The data folders should now look like this:
```
/path/to/xBD/train
├── guatemala-volcano
│   ├── images
│   ├── labels
│   └── masks
├── hurricane-florence
│   ├── images
│   ├── labels
│   └── masks
├── hurricane-harvey
│   ├── images
│   ├── labels
│   └── masks
├── hurricane-matthew
│   ├── images
│   ├── labels
│   └── masks
├── hurricane-michael
│   ├── images
│   ├── labels
│   └── masks
├── mexico-earthquake
│   ├── images
│   ├── labels
│   └── masks
├── midwest-flooding
│   ├── images
│   ├── labels
│   └── masks
├── palu-tsunami
│   ├── images
│   ├── labels
│   └── masks
├── santa-rosa-wildfire
│   ├── images
│   ├── labels
│   └── masks
├── socal-fire
│   ├── images
│   ├── labels
└   └── masks
```          

# Step 2: Investigate the data

In [ ]:
os.chdir(r"/content/xBD")

In [ ]:
all_image_paths = list(paths.list_images("train"))
print(f"Total images: {int(len(all_image_paths))/2}")

In [ ]:
image_ids = {path.split("/")[1] for path in all_image_paths}
print(len(image_ids))
image_ids

3. Implement a function `get_image_paths` that takes as an input the `image_id` and return the `image_paths` and `masks_paths`.

In [ ]:
### CODE HERE

4. Verify that for every pre-disaster image there exists a corresponding post-disaster image with the same tile ID (same geographic location). Report any missing pairs. Are these missing pairs uniformly distributed across disasters or concentrated in specific ones?

In case of error like `TypeError: Cannot convert numpy.ndarray to numpy.ndarray`, use [tabulate](https://github.com/astanin/python-tabulate) instead of [pd.DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html).


Hint: the below expression might suggest a usefull syntax:

```
int(len([f for f in image_paths if "_pre_" in f  ]))
```



In [ ]:
### CODE HERE

## Visualization

In [ ]:
pre_image_paths = [path for path in all_image_paths if ("images" in path)and ("_pre_" in path) and ("ipynb_checkpoints" not in path)]
post_image_paths = [path for path in all_image_paths if ("images" in path)and ("_post_" in path) and ("ipynb_checkpoints" not in path)]
pre_mask_label_paths = [path for path in all_image_paths if ("masks" in path) and ("_pre_" in path) and ("ipynb_checkpoints" not in path)]
post_mask_label_paths=[path for path in all_image_paths if ("masks" in path) and ("_post_" in path) and ("ipynb_checkpoints" not in path)]
len(pre_image_paths), len(post_image_paths), len(pre_mask_label_paths), len(post_mask_label_paths)

In [ ]:
regex = r"\D*\d+"
compiler = re.compile(regex)

def get_intensity(path):
  if sys.platform == 'win32':
    return compiler.search(path).group().split("\\")[-1]
  if sys.platform == 'linux':
    return compiler.search(path).group().split("/")[-1]

Before completing question 5, check the code `generate_annotation_masks.py` to familiarize with the structure of the labels. Hint: check the `mask_for_polygon function`.

5. Read the `pre_mask_label_paths` and `post_mask_label_paths` with [imageio.imread()](https://imageio.readthedocs.io/en/stable/_autosummary/imageio.v3.imread.html) and use [np.unique()](https://numpy.org/doc/stable/reference/generated/numpy.unique.html) to investigate how many value are usually present in the masks (both pre- and post-). Do you notice any incongruity between what you got and the Line 45-51 of the `generate_annotation_masks.py`:
```
damage_dict = {
    "no-damage": 1,
    "minor-damage": 2,
    "major-damage": 3,
    "destroyed": 4,
    "un-classified": 255
}
```

In [ ]:
### CODE HERE

In [ ]:
### ANSWER BELOW IN THE TEXT CELL

**6.** Write a `show_all_four_images` that displays pre-image, post-image, mask split by disaster class, and the post-disaster PNG with the damage mask overlaid as a semi-transparent layer. Plot a few random set of images. 

Requirements:
- All classes must each have a **distinct colour**; class 0 (background) must be **fully transparent** in the overlay.
- Overlay opacity: `alpha = 0.5`.
- Include a **legend** listing every class *present in this specific image* with its **class name and pixel count**.
- Use [`matplotlib.colors.ListedColormap`](https://matplotlib.org/stable/api/_as_gen/matplotlib.colors.ListedColormap.html).

Hint: to make background pixels transparent, build an RGBA array from the mask and set `rgba[mask == 0, 3] = 0`.
Hint: `mask_label_paths` includes both pre- and post-disaster images, use `post_mask_label_paths` for your for loop. A useful function can be `.replace()`, for example:

**Please ensure that ALL classes are displayed carefully.**



In [ ]:
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

DAMAGE_CLASSES = {
    0:   ('background',    'none'),
   ### CODE HERE
}

def show_all_four_images(pre_image_path, post_image_path, post_mask_path):
    """Display pre-image, post-image, colour-coded mask, and semi-transparent overlay.

    The fourth panel shows the post-disaster image with the damage mask overlaid
    at alpha=0.5.  Background pixels (class 0) are fully transparent in the overlay.
    A legend reports each damage class present in this image and its pixel count.

    Args:
        pre_image_path:  path to the pre-disaster PNG
        post_image_path: path to the post-disaster PNG
        post_mask_path:  path to the post-disaster mask file
    """
    ### CODE HERE


## Step 3: Class Distribution

7. Create a histogram illustrating the distribution of the pixels in the different classes.

In [ ]:
### CODE HERE

## Step 4: Building-level Quality Check

8. A building annotated in the **pre-disaster** JSON label should in principle also appear in the **post-disaster** JSON label (same geographic footprint, same tile). Implement `count_mismatched_buildings(disaster_name)` that:

   1. Finds all JSON files under `<disaster_name>/labels/`.
   2. For each **pre/post pair** sharing the same tile ID, parses the building polygon geometries from both JSONs.
   3. Computes the **centroid** (cx, cy) of each polygon in pixel coordinates, using the `"xy"` feature collection inside each JSON.
   4. Matches centroids between pre and post using a **pixel tolerance** (default `tol=10`) — two centroids are the same building if their Euclidean distance is below this threshold.
   5. Returns a dict `{tile_id: {"pre_only": int, "post_only": int}}` for every tile with at least one mismatch.

   Then iterate over **all disasters** and print a summary table of total mismatches per disaster.

   **What might cause mismatches?** Answer in the text cell below.

   ---
   **xBD JSON structure** (one entry in `features["xy"]`):

       {
         "properties": {"uid": "...", "subtype": "no-damage"},
         "wkt": "POLYGON ((x1 y1, x2 y2, ...))"
       }

   Parse the `"wkt"` string with [`shapely.wkt.loads()`](https://shapely.readthedocs.io/en/stable/manual.html) to get the polygon, then call `.centroid` on it.

   **Hint:** [`scipy.spatial.distance.cdist`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.cdist.html) computes all pairwise distances between two centroid arrays in one call — useful for matching step 4.

In [ ]:
import json
import glob
import numpy as np
from shapely import wkt as shapely_wkt
from scipy.spatial.distance import cdist


def parse_centroids(json_path):
    """Return an (N, 2) float array of building centroids in pixel coordinates.

    Each row is (cx, cy) for one building polygon found in the JSON label.
    Returns an empty array if the label contains no buildings.

    Args:
        json_path: path to an xBD label JSON file
    """
    ### CODE HERE


def count_mismatched_buildings(disaster_name, tol=10):
    """Count buildings present in pre- but not post-disaster labels (and vice versa).

    Args:
        disaster_name: name of the disaster folder (e.g. 'guatemala-volcano')
        tol: maximum pixel distance for two centroids to be considered the same
             building (default: 10 px)

    Returns:
        dict mapping tile_id -> {"pre_only": int, "post_only": int}
        Only tiles with at least one mismatch are included.
    """
    mismatches = {}
    label_dir = os.path.join(disaster_name, "labels")

    # Collect all pre-disaster JSON paths for this disaster
    pre_jsons = sorted(glob.glob(os.path.join(label_dir, "*_pre_disaster.json")))

    for pre_path in pre_jsons:
        ### CODE HERE
        # 1. Derive the post JSON path from pre_path (hint: use str.replace)
        # 2. Skip if the post JSON does not exist
        # 3. Parse centroids from both JSONs using parse_centroids()
        # 4. Use cdist to compute pairwise distances; a centroid is matched
        #    if its closest partner is within `tol` pixels
        # 5. Count unmatched centroids in each direction and store in mismatches
        pass

    return mismatches


# ── Run over all disasters and display a summary table ──────────────────────
disaster_names = [d for d in os.listdir(".") if os.path.isdir(d)]

### CODE HERE
# Iterate over disaster_names, call count_mismatched_buildings(),
# and print a summary: disaster | tiles with mismatches | total pre_only | total post_only


### ANSWER: What might cause building-level mismatches?

*Write your answer here.*